# Image Classification with AutoGluon

`MultiModalPredictor` can fine-tune pre-trained vision models (EfficientNet, ViT, ResNet via the TIMM library) on your labelled images with a single `.fit()` call.

This notebook covers:
1. Data format: image **file paths** in a DataFrame (not arrays or PIL Images)
2. Downloading a sample image dataset
3. Training a classifier and predicting
4. Swapping the backbone model
5. Practical gotchas — especially path handling

In [ ]:
import autogluon
print('AutoGluon version:', autogluon.__version__)

## 1. Download the Sample Dataset

AutoGluon provides a tiny shopee product dataset (images of clothing with category labels) for tutorials.

In [ ]:
import os
import zipfile
import urllib.request

DATA_URL = 'https://autogluon.s3.amazonaws.com/datasets/shopee-iet.zip'
DATA_DIR = './shopee_data'

if not os.path.exists(DATA_DIR):
    print('Downloading dataset...')
    zip_path = './shopee-iet.zip'
    urllib.request.urlretrieve(DATA_URL, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('.')
    os.rename('shopee-iet', DATA_DIR)
    os.remove(zip_path)
    print('Done.')
else:
    print('Dataset already present.')

# Inspect the directory structure
for split in ['train', 'test']:
    split_dir = os.path.join(DATA_DIR, 'data', split)
    classes = os.listdir(split_dir)
    print(f'{split}: {len(classes)} classes —', classes)

## 2. Build the DataFrame with Image Paths

**Critical:** AutoGluon's image column must contain **string file paths**, not PIL Image objects, not NumPy arrays, not bytes. It reads the images itself during training.

> **Always use absolute paths.** Relative paths are resolved from the Python process's working directory — which is NOT necessarily the notebook's directory. Using `os.path.abspath()` is safer.

In [ ]:
import pandas as pd

def build_dataframe(split_dir: str) -> pd.DataFrame:
    """Walk a directory of class-named subdirs and return a DataFrame of paths + labels."""
    rows = []
    for class_name in sorted(os.listdir(split_dir)):
        class_dir = os.path.join(split_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        for fname in os.listdir(class_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                rows.append({
                    'image': os.path.abspath(os.path.join(class_dir, fname)),  # absolute path!
                    'label': class_name,
                })
    return pd.DataFrame(rows)

train_df = build_dataframe(os.path.join(DATA_DIR, 'data', 'train'))
test_df  = build_dataframe(os.path.join(DATA_DIR, 'data', 'test'))

print(f'Train: {len(train_df)} images | Test: {len(test_df)} images')
print('Classes:', sorted(train_df['label'].unique()))
train_df.head()

In [ ]:
# Sanity check: ensure every file actually exists before fitting
missing = train_df[~train_df['image'].apply(os.path.exists)]
if len(missing) > 0:
    print(f'WARNING: {len(missing)} files not found!')
    print(missing['image'].head())
else:
    print('All image files exist.')

In [ ]:
# Preview a few images
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
sample = train_df.groupby('label').first().reset_index()
for ax, (_, row) in zip(axes, sample.iterrows()):
    ax.imshow(Image.open(row['image']))
    ax.set_title(row['label'])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Train the Classifier

In [ ]:
from autogluon.multimodal import MultiModalPredictor

predictor = MultiModalPredictor(
    label='label',
    problem_type='multiclass',
    path='./ag_image_model',
)

predictor.fit(
    train_data=train_df,
    time_limit=120,
)

## 4. Predict and Evaluate

In [ ]:
y_pred = predictor.predict(test_df)
metrics = predictor.evaluate(test_df)
print('Test metrics:', metrics)

y_prob = predictor.predict_proba(test_df)
print('\nProbability columns:', y_prob.columns.tolist())

In [ ]:
# Visualise predictions on a few test images
sample_test = test_df.sample(4, random_state=42).reset_index(drop=True)
preds = predictor.predict(sample_test)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (_, row), pred in zip(axes, sample_test.iterrows(), preds):
    ax.imshow(Image.open(row['image']))
    colour = 'green' if pred == row['label'] else 'red'
    ax.set_title(f'True: {row["label"]}\nPred: {pred}', color=colour, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Choosing a Different Backbone

AutoGluon uses the [TIMM](https://github.com/huggingface/pytorch-image-models) library for vision backbones. You can pass any TIMM model name.

Common choices:
- `'swin_small_patch4_window7_224'` — modern, accurate Swin Transformer
- `'efficientnet_b0'` — fast and compact
- `'vit_base_patch16_224'` — classic Vision Transformer
- `'resnet50'` — solid baseline

In [ ]:
# Example: switch to EfficientNet-B0 (faster, smaller)
fast_predictor = MultiModalPredictor(
    label='label',
    problem_type='multiclass',
    path='./ag_image_efficientnet',
)
fast_predictor.fit(
    train_data=train_df,
    time_limit=120,
    hyperparameters={
        'model.timm_image.checkpoint_name': 'efficientnet_b0',
    },
)
print('EfficientNet-B0 metrics:', fast_predictor.evaluate(test_df))

## ⚠️ Practical Gotchas

| Gotcha | What Happens | Fix |
|--------|-------------|-----|
| **Relative paths in DataFrame** | Path works at fit time but breaks if you call predict() from a different working directory | Always use `os.path.abspath()` when building the DataFrame |
| **Passing PIL Image objects** | `TypeError` — AutoGluon expects string paths, not objects | Save images to disk and pass their paths |
| **Missing image files** | Cryptic error mid-training after the epoch starts | Validate all paths with `os.path.exists()` before calling `.fit()` |
| **Wrong file extension** | A `.jpg` file that is actually a PNG causes decode errors | Use `imghdr` or `Pillow` to verify actual format matches extension |
| **Column name auto-detection** | AutoGluon inspects values, not column names, to decide if a column is image | Column values must be valid file paths ending in an image extension |
| **Huge images (>4000px)** | OOM on GPU during augmentation | Resize to ≤1024px before training |
| **Unbalanced classes** | Minority classes get poor recall | Oversample minority classes or pass `hyperparameters={'data.pos_weight': ...}` |
| **GPU memory error** | Batch too large for VRAM | Reduce batch size: `hyperparameters={'env.per_gpu_batch_size': 8}` |